In [ ]:
# @title ▶ Setup ও App চালু করুন · ইংরেজি ভিডিও → বাংলা
# @markdown **T4 GPU** বেছে ▶ চাপুন। ইনপুট: **My Drive / Video / input** → ফলাফল: **Video / output**।
# @markdown প্রথমবার Drive-এর অনুমতি ও model download লাগবে। আবার চালাতেও এই একটি cell যথেষ্ট।
RUN_RESULT = None
REMOUNT_DRIVE = False # @param {type:"boolean"}
import sys
import shutil
import subprocess
import runpy
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10 বা নতুন সংস্করণ প্রয়োজন।")
try:
    import google.colab
except ImportError as error:
    raise RuntimeError("এই notebook Google Colab-এ খুলুন।") from error

APP_DIR = Path("/content/proxmax-app")
APP_DIR.mkdir(parents=True, exist_ok=True)
SOURCES = {'colab_runtime.py': '"""Colab setup ও আলাদা worker চালানোর সহায়ক কোড; এখানে ML import করা হয় না।"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport sys\nimport uuid\n\n\ndef setup_environment(root, requirements):\n    """Colab-এর প্যাকেজ না বদলে নিজস্ব venv বানায়; সফল setup পুনরায় ব্যবহার করে।"""\n    root = Path(root).resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    env = os.environ.copy()\n    env.pop("PYTHONPATH", None)\n    env.pop("PYTHONHOME", None)\n    environment = root / "venv"\n    python = environment / "bin" / "python"\n    marker = root / "environment.json"\n    # Colab-এর pip ও torch দেখা যাবে; নতুন প্যাকেজ কেবল এই venv-এ লেখা হবে।\n    # --without-pip ব্যবহার করায় Debian/Colab-এর ensurepip দরকার হয় না।\n    if not python.exists():\n        subprocess.run([sys.executable, "-m", "venv", "--system-site-packages",\n                        "--without-pip", str(environment)], check=True, env=env)\n    requirement_file = root / "requirements.txt"\n    requirement_file.write_text(requirements, encoding="utf-8")\n    fingerprint = hashlib.sha256(\n        (requirements + sys.version + sys.executable).encode()).hexdigest()\n    # Parent kernel-এর import cache নয়, নতুন interpreter-এর প্যাকেজ যাচাই করা হয়।\n    check_code = """\nimport importlib.metadata as metadata\nimport sys\nfrom packaging.requirements import Requirement\nfrom packaging.version import Version\nif Version(metadata.version(\'torch\').split(\'+\')[0]) < Version(\'2.6\'):\n    raise SystemExit(\'torch >= 2.6 প্রয়োজন; বর্তমান Colab runtime ব্যবহার করুন।\')\nfor line in open(sys.argv[1], encoding=\'utf-8\'):\n    line = line.strip()\n    if not line or line.startswith(\'#\'):\n        continue\n    requirement = Requirement(line)\n    if requirement.marker and not requirement.marker.evaluate():\n        continue\n    if not requirement.specifier.contains(metadata.version(requirement.name)):\n        raise SystemExit(1)\n"""\n    ready = False\n    if marker.exists():\n        try:\n            ready = json.loads(marker.read_text())["fingerprint"] == fingerprint\n        except (ValueError, KeyError):\n            pass\n    if ready:\n        ready = subprocess.run([str(python), "-c", check_code, str(requirement_file)],\n                               capture_output=True, text=True, env=env).returncode == 0\n    if not ready:\n        marker.unlink(missing_ok=True)\n        print("প্রথমবারের প্যাকেজ setup চলছে…", flush=True)\n        subprocess.run([str(python), "-m", "pip", "--isolated", "install", "--no-user", "--quiet",\n                        "--disable-pip-version-check", "-r", str(requirement_file)], check=True, env=env)\n        subprocess.run([str(python), "-c", check_code, str(requirement_file)], check=True, env=env)\n        temp = marker.with_suffix(".partial.json")\n        temp.write_text(json.dumps({"fingerprint": fingerprint}), encoding="utf-8")\n        os.replace(temp, marker)\n    else:\n        print("আগের setup ব্যবহার করা হচ্ছে; আবার install লাগবে না।")\n    return python\n\n\ndef ensure_drive_mounted(mount_point="/content/drive", force_remount=False):\n    """সক্রিয় mount পুনর্ব্যবহার করে; শুধু directory থাকাকে mount ধরে না।"""\n    mount_point = Path(mount_point)\n    root = mount_point / "MyDrive"\n    try:\n        mounted = os.path.ismount(mount_point)\n        accessible = root.is_dir() if mounted else False\n        if accessible:\n            # Stale FUSE mount-এ stat সফল হলেও directory পড়া ব্যর্থ হতে পারে।\n            next(root.iterdir(), None)\n    except OSError as error:\n        if not force_remount:\n            raise RuntimeError("Drive সংযোগে সমস্যা। REMOUNT_DRIVE=True করে এই cell আবার চালান।") from error\n        mounted = accessible = False\n    if not force_remount and mounted:\n        if not accessible:\n            raise RuntimeError("MyDrive পাওয়া যাচ্ছে না। REMOUNT_DRIVE=True করে এই cell আবার চালান।")\n        return root\n    from google.colab import drive\n    drive.mount(str(mount_point), force_remount=force_remount)\n    if not os.path.ismount(mount_point) or not root.is_dir():\n        raise RuntimeError("Drive mount হয়নি। Google-এর অনুমতি দিয়ে আবার চেষ্টা করুন।")\n    return root\n\n\ndef stop_worker(process, timeout=5):\n    """Interrupt হলে worker এবং তার FFmpeg subprocess-গুলোও বন্ধ করে।"""\n    def send(sig):\n        try:\n            os.killpg(process.pid, sig)\n        except ProcessLookupError:\n            pass\n\n    send(signal.SIGTERM)\n    try:\n        process.wait(timeout=timeout)\n    except subprocess.TimeoutExpired:\n        send(signal.SIGKILL)\n        process.wait()\n    finally:\n        # Parent আগে থামলেও group-এর কোনো FFmpeg যেন চলতে না থাকে।\n        send(signal.SIGKILL)\n\n\ndef run_isolated(python, worker, config, state_dir, echo=True):\n    """প্রতিবার নতুন Python/CUDA process; আগের run-এর GPU/import state ধরে রাখে না।"""\n    state_dir = Path(state_dir).resolve()\n    state_dir.mkdir(parents=True, exist_ok=True)\n    run_id = uuid.uuid4().hex\n    config_path = state_dir / f"{run_id}.config.json"\n    results_path = state_dir / f"{run_id}.results.json"\n    log_path = state_dir / f"{run_id}.log"\n    config_path.write_text(json.dumps(config, ensure_ascii=False), encoding="utf-8")\n    env = os.environ.copy()\n    env.update(PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false",\n               HF_HOME="/content/proxmax-model-cache", HF_HUB_DISABLE_TELEMETRY="1")\n    # আগের notebook থেকে পাওয়া PYTHONPATH/PYTHONHOME যেন venv isolation নষ্ট না করে।\n    env.pop("PYTHONPATH", None)\n    env.pop("PYTHONHOME", None)\n    process = subprocess.Popen(\n        [str(python), str(Path(worker).resolve()), "--config", str(config_path),\n         "--results", str(results_path)],\n        stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n        text=True, encoding="utf-8", errors="replace", env=env, start_new_session=True,\n    )\n    try:\n        with log_path.open("w", encoding="utf-8") as log:\n            for line in process.stdout:\n                log.write(line)\n                log.flush()\n                if echo:\n                    print(line, end="", flush=True)\n        returncode = process.wait()\n    except BaseException:\n        stop_worker(process)\n        raise\n    finally:\n        process.stdout.close()\n    try:\n        outcome = json.loads(results_path.read_text(encoding="utf-8"))\n    except (OSError, ValueError):\n        outcome = {"status": "failed", "results": [],\n                   "error": f"Worker বন্ধ হয়েছে (code {returncode})। নিচের log দেখুন, তারপর আবার run করুন।"}\n    if returncode:\n        outcome["status"] = "failed"\n    return {**outcome, "returncode": returncode, "log": str(log_path)}\n', 'pipeline.py': '"""Local-first English-to-Bangla dubbing. Heavy dependencies are imported by stage.\n\nThis file is also the source for the standalone Colab notebook. After editing it,\nrun `python scripts/build_notebook.py` and commit both files.\n"""\n\n# %% Configuration and standard-library utilities\nfrom __future__ import annotations\n\nimport ctypes\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport logging\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport uuid\nimport wave\nfrom dataclasses import asdict, dataclass\nfrom typing import Callable\n\nPIPELINE_VERSION = "1.0.0"\nVIDEO_EXTENSIONS = {".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v"}\nLOG = logging.getLogger("proxmax")\n\n\n@dataclass(frozen=True)\nclass Config:\n    input_dir: str = "/content/drive/MyDrive/Video/input"\n    output_dir: str = "/content/drive/MyDrive/Video/output"\n    work_dir: str = "/content/proxmax-work"\n    # These are public, downloadable checkpoints, not hosted inference APIs.\n    asr_model: str = "Systran/faster-whisper-large-v3-turbo"\n    translation_model: str = "facebook/nllb-200-distilled-600M"\n    tts_model: str = "facebook/mms-tts-ben"\n    # Replace \'main\' with Hub commit IDs for reproducible model snapshots.\n    asr_revision: str = "main"\n    translation_revision: str = "main"\n    tts_revision: str = "main"\n    device: str = "auto"  # auto, cuda, or cpu\n    asr_beam_size: int = 5\n    translation_beams: int = 3\n    translation_batch_size: int = 4\n    cpu_threads: int = 2\n    max_utterance_seconds: float = 12.0\n    max_utterance_words: int = 40\n    split_gap_seconds: float = 0.65\n    borrow_silence_seconds: float = 0.30\n    comfortable_speed: float = 1.50\n    overflow_policy: str = "stretch"  # stretch and flag, or error (never cut words)\n    max_source_tokens: int = 480\n    max_target_tokens: int = 512\n    max_tts_chars: int = 220\n    sample_rate: int = 16000\n    seed: int = 42\n    # Successful jobs are removed locally only AFTER verified publication.\n    keep_local: bool = False\n    continue_on_error: bool = True\n    min_free_gb: float = 6.0\n    subprocess_timeout: int = 7200\n\n    def validate(self):\n        if self.device not in {"auto", "cpu", "cuda"}:\n            raise ValueError("device must be auto, cpu, or cuda")\n        if self.overflow_policy not in {"stretch", "error"}:\n            raise ValueError("overflow_policy must be stretch or error")\n        for name in ("asr_beam_size", "translation_beams", "translation_batch_size",\n                     "cpu_threads", "max_utterance_seconds", "max_utterance_words",\n                     "comfortable_speed", "max_source_tokens", "max_target_tokens",\n                     "max_tts_chars", "sample_rate", "subprocess_timeout"):\n            value = getattr(self, name)\n            if not math.isfinite(value) or value <= 0:\n                raise ValueError(f"{name} must be finite and positive")\n        if self.comfortable_speed < 1:\n            raise ValueError("comfortable_speed must be at least 1")\n        for name in ("borrow_silence_seconds", "split_gap_seconds", "min_free_gb"):\n            if not math.isfinite(getattr(self, name)) or getattr(self, name) < 0:\n                raise ValueError(f"{name} must be finite and nonnegative")\n        source, destination, work = map(\n            lambda p: Path(p).expanduser().resolve(),\n            (self.input_dir, self.output_dir, self.work_dir),\n        )\n        if source == destination or source in destination.parents or destination in source.parents:\n            raise ValueError("Input and output directories must be separate, non-nested folders")\n        for drive_path in (source, destination):\n            if work == drive_path or work in drive_path.parents or drive_path in work.parents:\n                raise ValueError("Work directory must be separate from input/output")\n        if str(work).startswith("/content/drive/"):\n            raise ValueError("Work directory must be local, not on mounted Google Drive")\n\n\ndef run_command(args, timeout=7200):\n    """No shell interpolation: filenames may contain spaces, Unicode, or quotes."""\n    result = subprocess.run([str(x) for x in args], capture_output=True, text=True,\n                            timeout=timeout, check=False)\n    if result.returncode:\n        raise RuntimeError(f"{args[0]} failed ({result.returncode}):\\n{result.stderr[-6000:]}")\n    return result.stdout\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + ".partial-" + uuid.uuid4().hex)\n    try:\n        with temp.open("w", encoding="utf-8") as stream:\n            json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)\n            stream.flush()\n            os.fsync(stream.fileno())\n        os.replace(temp, path)\n    finally:\n        temp.unlink(missing_ok=True)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef atomic_copy(source, destination):\n    """Publish to a temporary sibling, verify bytes, then rename on Drive\'s mount."""\n    source, destination = Path(source), Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    expected = sha256_file(source)\n    if destination.is_file() and sha256_file(destination) == expected:\n        return expected\n    temp = destination.with_name(destination.name + ".partial-" + uuid.uuid4().hex)\n    try:\n        shutil.copyfile(source, temp)\n        if sha256_file(temp) != expected:\n            raise IOError(f"Checksum mismatch while publishing {destination}")\n        os.replace(temp, destination)\n    finally:\n        temp.unlink(missing_ok=True)\n    return expected\n\n\ndef safe_stem(name):\n    # Limit bytes as well as characters so long Unicode filenames remain portable.\n    stem = re.sub(r"[^\\w.-]+", "_", Path(name).stem, flags=re.UNICODE).strip("._") or "video"\n    return stem.encode("utf-8")[:100].decode("utf-8", errors="ignore") or "video"\n\n\ndef package_versions():\n    versions = {}\n    for name in ("torch", "transformers", "faster-whisper", "ctranslate2", "uroman",\n                 "numpy", "soundfile", "sentencepiece"):\n        try:\n            versions[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            versions[name] = "not-installed"\n    return versions\n\n\ndef recipe_id(config):\n    recipe = asdict(config)\n    for key in ("input_dir", "output_dir", "work_dir", "keep_local", "continue_on_error",\n                "min_free_gb", "subprocess_timeout"):\n        recipe.pop(key)\n    encoded = json.dumps({"pipeline": PIPELINE_VERSION, "config": recipe,\n                          "packages": package_versions()}, sort_keys=True).encode()\n    return hashlib.sha256(encoded).hexdigest()[:12]\n\n\n# %% Local staging, probing and media extraction\n\ndef probe(path, timeout=120):\n    return json.loads(run_command([\n        "ffprobe", "-v", "error", "-show_format", "-show_streams", "-of", "json", path\n    ], timeout))\n\n\ndef media_info(path):\n    data = probe(path)\n    videos = [s for s in data["streams"] if s["codec_type"] == "video"\n              and not s.get("disposition", {}).get("attached_pic", 0)]\n    audios = [s for s in data["streams"] if s["codec_type"] == "audio"]\n    if not videos:\n        raise ValueError(f"No video stream in {Path(path).name}")\n    duration = float(data.get("format", {}).get("duration", 0))\n    if not math.isfinite(duration) or duration <= 0:\n        raise ValueError("Input must have a finite, positive duration")\n    return {"duration": duration, "video_index": videos[0]["index"],\n            "audio_index": audios[0]["index"] if audios else None,\n            "video_codec": videos[0]["codec_name"], "probe": data}\n\n\ndef stage_input(source, config):\n    """Read Drive once for staging; every subsequent media operation is local."""\n    root = Path(config.work_dir)\n    root.mkdir(parents=True, exist_ok=True)\n    size = source.stat().st_size\n    # Reserve for source, rendered video, PCM/intermediates and model downloads.\n    required = size * 3 + config.min_free_gb * 1024**3\n    if shutil.disk_usage(root).free < required:\n        raise OSError("Insufficient local disk: free space or increase Colab storage")\n    fd, name = tempfile.mkstemp(prefix="staging-", suffix=source.suffix.lower(), dir=root)\n    digest = hashlib.sha256()\n    before = source.stat()\n    try:\n        with os.fdopen(fd, "wb") as out, source.open("rb") as inp:\n            for block in iter(lambda: inp.read(8 * 1024 * 1024), b""):\n                out.write(block)\n                digest.update(block)\n        after = source.stat()\n        if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):\n            raise IOError("Input changed while copying; retry when its upload has finished")\n        job_id = f"{safe_stem(source.name)}-{digest.hexdigest()[:12]}-{recipe_id(config)}"\n        job = root / job_id\n        job.mkdir(exist_ok=True)\n        local = job / ("input" + source.suffix.lower())\n        os.replace(name, local)\n        return job, local, digest.hexdigest()\n    finally:\n        Path(name).unlink(missing_ok=True)\n\n\ndef extract_audio(local_video, destination, info, config):\n    if info["audio_index"] is None:\n        raise ValueError("Input has no audio stream to translate; add/select a video with English speech")\n    temp = Path(destination).with_suffix(".partial.wav")\n    run_command([\n        "ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",\n        "-i", local_video, "-map", f"0:{info[\'audio_index\']}", "-vn",\n        "-af", "aresample=16000:async=1:first_pts=0", "-ac", "1", "-ar", "16000",\n        "-t", str(info["duration"]), "-c:a", "pcm_s16le", temp,\n    ], config.subprocess_timeout)\n    os.replace(temp, destination)\n\n\n# %% Timestamp grouping and subtitle generation (no ML dependencies)\n\ndef group_words(words, duration, config):\n    """Keep absolute ASR times; split at sentences, pauses and bounded phrase lengths."""\n    output, group = [], []\n    previous_end = 0.0\n\n    def flush():\n        if group:\n            output.append({"id": len(output), "start": group[0]["start"],\n                           "end": group[-1]["end"],\n                           "english": " ".join(w["text"] for w in group).strip()})\n            group.clear()\n\n    for word in words:\n        text = str(word.get("text", "")).strip()\n        start, end = float(word["start"]), float(word["end"])\n        if not math.isfinite(start) or not math.isfinite(end):\n            raise ValueError("Nonfinite ASR timestamp")\n        start, end = max(0.0, previous_end, start), min(duration, end)\n        if not text or end <= start:\n            continue\n        if group and (\n            start - group[-1]["end"] >= config.split_gap_seconds\n            or end - group[0]["start"] > config.max_utterance_seconds\n            or len(group) >= config.max_utterance_words\n            or re.search(r"[.!?][\\"\')\\]]*$", group[-1]["text"])\n        ):\n            flush()\n        group.append({"start": start, "end": end, "text": text})\n        previous_end = end\n    flush()\n    return output\n\n\ndef validate_segments(segments, duration, translated=False):\n    previous_end = 0.0\n    for index, segment in enumerate(segments):\n        start, end = float(segment["start"]), float(segment["end"])\n        if (segment["id"] != index or not math.isfinite(start) or not math.isfinite(end)\n                or start < previous_end - 1e-6 or end <= start or end > duration + 1e-6):\n            raise ValueError(f"Invalid/nonmonotonic segment {index}")\n        if not str(segment["english"]).strip():\n            raise ValueError(f"Empty English text in segment {index}")\n        if translated and not str(segment.get("bangla", "")).strip():\n            raise ValueError(f"Empty Bangla translation in segment {index}")\n        previous_end = end\n\n\ndef srt_time(seconds):\n    ms = max(0, round(seconds * 1000))\n    hours, ms = divmod(ms, 3_600_000)\n    minutes, ms = divmod(ms, 60_000)\n    secs, ms = divmod(ms, 1000)\n    return f"{hours:02}:{minutes:02}:{secs:02},{ms:03}"\n\n\ndef write_srt(segments, key, path):\n    lines = []\n    for i, segment in enumerate(segments, 1):\n        # Bengali subtitles cover the actual allowed dubbing slot, including borrowed silence.\n        end = segment.get("slot_end", segment["end"]) if key == "bangla" else segment["end"]\n        lines.append(f"{i}\\n{srt_time(segment[\'start\'])} --> {srt_time(end)}\\n{segment[key]}\\n")\n    Path(path).write_text("\\n".join(lines), encoding="utf-8")\n\n\ndef atempo_filter(rate):\n    if not math.isfinite(rate) or rate <= 0:\n        raise ValueError("Tempo must be finite and positive")\n    factors = []\n    while rate > 2.0:\n        factors.append(2.0)\n        rate /= 2.0\n    while rate < 0.5:\n        factors.append(0.5)\n        rate /= 0.5\n    factors.append(rate)\n    return ",".join(f"atempo={f:.10f}" for f in factors)\n\n\ndef split_tts_text(text, limit):\n    """Bound VITS memory use without dropping words or splitting Unicode codepoints."""\n    chunks, current = [], ""\n    for word in text.split():\n        if len(word) > limit:\n            raise ValueError("A translation contains an overlong token; edit it before synthesis")\n        if current and len(current) + len(word) + 1 > limit:\n            chunks.append(current)\n            current = ""\n        current = (current + " " + word).strip()\n        if current.endswith(("।", ".", "?", "!")):\n            chunks.append(current)\n            current = ""\n    if current:\n        chunks.append(current)\n    if not chunks:\n        raise ValueError("Cannot synthesize empty text")\n    return chunks\n\n\n# %% Device setup and English speech recognition\n\ndef torch_device(config):\n    import torch\n    torch.set_num_threads(config.cpu_threads)\n    if config.device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA requested but unavailable; select a Colab GPU or device=\'cpu\'")\n    return "cuda" if config.device != "cpu" and torch.cuda.is_available() else "cpu"\n\n\ndef release_memory():\n    gc.collect()\n    if "torch" in sys.modules:\n        import torch\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n\ndef preload_cuda_libraries():\n    """Help CTranslate2 locate Colab/PyTorch\'s CUDA 12/cuDNN 9 shared libraries."""\n    import site\n    roots = [Path(p) / "nvidia" for p in site.getsitepackages()]\n    for package, library in (("cublas", "libcublasLt.so.12"),\n                             ("cublas", "libcublas.so.12"), ("cudnn", "libcudnn.so.9")):\n        for root in roots:\n            candidate = root / package / "lib" / library\n            if candidate.is_file():\n                ctypes.CDLL(str(candidate), mode=ctypes.RTLD_GLOBAL)\n                break\n\n\ndef transcribe(audio_path, duration, config):\n    import ctranslate2\n    from faster_whisper import WhisperModel\n    from huggingface_hub import snapshot_download\n    device = torch_device(config)\n    if device == "cuda":\n        preload_cuda_libraries()\n    supported = ctranslate2.get_supported_compute_types(device)\n    compute_type = "int8_float16" if device == "cuda" else "int8"\n    if compute_type not in supported:\n        compute_type = "float16" if device == "cuda" else "float32"\n    LOG.info("ইংরেজি শনাক্তকরণ: %s (%s / %s)", config.asr_model, device, compute_type)\n    model = None\n    try:\n        model_path = snapshot_download(config.asr_model, revision=config.asr_revision,\n                                       allow_patterns=["*.json", "*.bin", "*.txt"])\n        model = WhisperModel(model_path, device=device, compute_type=compute_type,\n                             cpu_threads=config.cpu_threads, num_workers=1)\n        # Non-batched decoding is deliberately conservative on free-tier RAM/VRAM.\n        stream, info = model.transcribe(\n            str(audio_path), language="en", task="transcribe",\n            beam_size=config.asr_beam_size, word_timestamps=True,\n            vad_filter=True, vad_parameters={"min_silence_duration_ms": 500},\n            condition_on_previous_text=False,\n        )\n        words, raw_segments = [], []\n        for segment in stream:  # Consume the lazy generator while the model is alive.\n            raw_segments.append({"start": segment.start, "end": segment.end,\n                                 "text": segment.text, "avg_logprob": segment.avg_logprob,\n                                 "no_speech_prob": segment.no_speech_prob})\n            if segment.words:\n                words.extend({"start": w.start, "end": w.end, "text": w.word,\n                              "probability": w.probability} for w in segment.words)\n            elif segment.text.strip():\n                words.append({"start": segment.start, "end": segment.end,\n                              "text": segment.text})\n        segments = group_words(words, duration, config)\n        validate_segments(segments, duration)\n        return {"segments": segments, "words": words, "raw_segments": raw_segments,\n                "language": "en", "device": device, "compute_type": compute_type,\n                "model_commit": Path(model_path).name,\n                "duration_after_vad": info.duration_after_vad}\n    finally:\n        del model\n        release_memory()\n\n\n# %% Batched English-to-Bangla translation\n\ndef translate(segments, config):\n    if not segments:\n        return {"segments": [], "model_commit": None}\n    import torch\n    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer\n    device = torch_device(config)\n    tokenizer = model = inputs = generated = None\n    try:\n        tokenizer = AutoTokenizer.from_pretrained(\n            config.translation_model, revision=config.translation_revision,\n            src_lang="eng_Latn", use_fast=True, trust_remote_code=False,\n        )\n        model = AutoModelForSeq2SeqLM.from_pretrained(\n            config.translation_model, revision=config.translation_revision,\n            torch_dtype=torch.float16 if device == "cuda" else torch.float32,\n            trust_remote_code=False,\n        ).to(device).eval()\n        bangla_id = tokenizer.convert_tokens_to_ids("ben_Beng")\n        if bangla_id is None or bangla_id == tokenizer.unk_token_id:\n            raise ValueError("Translation tokenizer does not support ben_Beng")\n        output = []\n        for offset in range(0, len(segments), config.translation_batch_size):\n            batch = segments[offset:offset + config.translation_batch_size]\n            texts = [s["english"] for s in batch]\n            lengths = [len(tokenizer.encode(t)) for t in texts]\n            if max(lengths) > config.max_source_tokens:\n                raise ValueError("ASR phrase exceeds translation token limit; reduce max_utterance_words")\n            inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=False).to(device)\n            with torch.inference_mode():\n                generated = model.generate(\n                    **inputs, forced_bos_token_id=bangla_id,\n                    num_beams=config.translation_beams, max_new_tokens=config.max_target_tokens,\n                )\n            for segment, ids in zip(batch, generated):\n                # NLLB\'s decoder start token can itself be EOS: ignore the first token.\n                nonpadding = [t for t in ids.tolist() if t != tokenizer.pad_token_id]\n                if (tokenizer.eos_token_id not in ids[1:].tolist()\n                        or len(nonpadding) >= config.max_target_tokens + 1):\n                    # A forced EOS at max_length is still potentially truncated text.\n                    raise ValueError(f"Translation hit the generation limit at segment {segment[\'id\']}")\n                text = tokenizer.decode(ids, skip_special_tokens=True).strip()\n                if not text or not re.search(r"[\\u0980-\\u09FF]", text):\n                    raise ValueError(f"Empty/non-Bengali output at segment {segment[\'id\']}; review source text")\n                output.append({**segment, "bangla": text})\n            LOG.info("অনুবাদ হয়েছে: %s/%s অংশ", len(output), len(segments))\n        return {"segments": output, "model_commit": getattr(model.config, "_commit_hash", None)}\n    finally:\n        del inputs, generated, model, tokenizer\n        release_memory()\n\n\n# %% Local Bengali speech synthesis and duration fitting\n\ndef fit_audio(raw_path, output_path, slot_seconds, config):\n    """Speed up only when needed; never truncate spoken samples to force a fit."""\n    import numpy as np\n    import soundfile as sf\n    native = sf.info(raw_path)\n    raw_seconds = native.frames / native.samplerate\n    target_samples = round(slot_seconds * config.sample_rate)\n    if target_samples <= 0 or raw_seconds <= 0:\n        raise ValueError("Audio and timestamp slot must have positive duration")\n    rate = max(1.0, raw_seconds / slot_seconds)\n    flagged = rate > config.comfortable_speed\n    if flagged and config.overflow_policy == "error":\n        raise ValueError(f"Dub needs {rate:.2f}x speed; edit/shorten translation or use overflow_policy=\'stretch\'")\n    if flagged:\n        LOG.warning("উচ্চারণ %.2fx দ্রুত হচ্ছে (সীমা %.2fx); শুনে যাচাই করুন",\n                    rate, config.comfortable_speed)\n    temp = Path(output_path).with_suffix(".tempo.wav")\n    try:\n        for attempt in range(4):\n            # Resample before tempo so target counts refer to the timeline sample rate.\n            run_command([\n                "ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",\n                "-i", raw_path, "-af", f"aresample={config.sample_rate}," + atempo_filter(rate),\n                "-ar", str(config.sample_rate), "-ac", "1", "-c:a", "pcm_s16le", temp,\n            ], config.subprocess_timeout)\n            samples, sr = sf.read(temp, dtype="float32")\n            if len(samples) <= target_samples:\n                break\n            # atempo is not sample-exact. Retry faster instead of cropping phonemes.\n            rate *= (len(samples) / target_samples) * 1.015\n            if config.overflow_policy == "error" and rate > config.comfortable_speed:\n                raise ValueError("Sample-exact fit would exceed comfortable_speed; edit the translation")\n        else:\n            raise RuntimeError("Could not fit speech without truncation; edit this translation")\n        if not len(samples) or not np.isfinite(samples).all() or np.max(np.abs(samples)) < 1e-5:\n            raise ValueError("Synthesis produced empty, silent or nonfinite audio")\n        # Pad silence, never speech-crop. This also gives exact per-slot length.\n        samples = np.pad(samples, (0, target_samples - len(samples)))\n        partial = Path(output_path).with_suffix(".partial.wav")\n        sf.write(partial, samples, sr, subtype="PCM_16")\n        os.replace(partial, output_path)\n        return {"raw_seconds": raw_seconds, "speed_factor": rate,\n                "review_required": rate > config.comfortable_speed,\n                "slot_seconds": target_samples / config.sample_rate}\n    finally:\n        temp.unlink(missing_ok=True)\n\n\ndef synthesize(segments, duration, job, config):\n    import numpy as np\n    import soundfile as sf\n    import torch\n    from transformers import AutoTokenizer, VitsModel, set_seed\n    from uroman import Uroman\n    device = torch_device(config)\n    clips_dir = Path(job) / "clips"\n    clips_dir.mkdir(exist_ok=True)\n    tokenizer = model = romanizer = tokens = result = None\n    records = []\n    recipe = recipe_id(config)\n    tts_commit = None\n    try:\n        for i, segment in enumerate(segments):\n            next_start = segments[i + 1]["start"] if i + 1 < len(segments) else duration\n            slot_end = min(duration, next_start, segment["end"] + config.borrow_silence_seconds)\n            start_sample = round(segment["start"] * config.sample_rate)\n            end_sample = round(slot_end * config.sample_rate)\n            slot_seconds = (end_sample - start_sample) / config.sample_rate\n            if slot_seconds <= 0:\n                raise ValueError(f"Segment {i} has no available audio slot")\n            signature = hashlib.sha256(json.dumps(\n                {"segment": segment, "slot_end": slot_end, "recipe": recipe},\n                sort_keys=True, ensure_ascii=False,\n            ).encode()).hexdigest()[:16]\n            clip = clips_dir / f"{i:05d}-{signature}.wav"\n            metadata = clip.with_suffix(".json")\n            if clip.exists() and metadata.exists():\n                record = read_json(metadata)\n                if record.get("sha256") == sha256_file(clip):\n                    records.append(record)\n                    tts_commit = record.get("tts_model_commit", tts_commit)\n                    continue\n            if model is None:\n                tokenizer = AutoTokenizer.from_pretrained(\n                    config.tts_model, revision=config.tts_revision, trust_remote_code=False)\n                model = VitsModel.from_pretrained(\n                    config.tts_model, revision=config.tts_revision,\n                    trust_remote_code=False,\n                ).to(device).eval()  # FP32 is deliberate: VITS can be unstable in FP16.\n                tts_commit = getattr(model.config, "_commit_hash", None)\n                if getattr(tokenizer, "is_uroman", False):\n                    romanizer = Uroman()\n                    # Reuse one Romanizer, rather than construct one for every tokenization.\n                    tokenizer.is_uroman = False\n            set_seed(config.seed + i)\n            parts = []\n            native_rate = model.config.sampling_rate\n            for text in split_tts_text(segment["bangla"], config.max_tts_chars):\n                if romanizer is not None:\n                    text = romanizer.romanize_string(text, lcode="ben")\n                tokens = tokenizer(text, return_tensors="pt").to(device)\n                if tokens.input_ids.shape[-1] <= 1:\n                    raise ValueError(f"TTS tokenizer has no pronounceable text in segment {i}")\n                with torch.inference_mode():\n                    result = model(**tokens).waveform\n                part = result[0].float().cpu().numpy()\n                if not len(part) or not np.isfinite(part).all():\n                    raise ValueError(f"Invalid TTS waveform in segment {i}")\n                if parts:\n                    parts.append(np.zeros(round(0.08 * native_rate), dtype=np.float32))\n                parts.append(part)\n            audio = np.concatenate(parts)\n            peak = float(np.max(np.abs(audio)))\n            if peak > 0:\n                audio *= min(1.0, 0.95 / peak)  # Prevent clipping without boosting noise.\n            raw = clips_dir / f"{i:05d}-raw.wav"\n            sf.write(raw, audio, native_rate, subtype="PCM_16")\n            timing = fit_audio(raw, clip, slot_seconds, config)\n            record = {**segment, **timing, "slot_end": end_sample / config.sample_rate,\n                      "start_sample": start_sample, "clip": clip.name,\n                      "sha256": sha256_file(clip), "tts_model_commit": tts_commit}\n            atomic_json(metadata, record)\n            raw.unlink(missing_ok=True)\n            records.append(record)\n            LOG.info("বাংলা অডিও তৈরি: %s/%s অংশ", i + 1, len(segments))\n        return records, tts_commit\n    finally:\n        del tokens, result, model, tokenizer, romanizer\n        release_memory()\n\n\n# %% Disk-backed audio timeline and final video muxing\n\ndef assemble_timeline(records, duration, job, config):\n    import numpy as np\n    import soundfile as sf\n    job = Path(job)\n    total_samples = math.ceil(duration * config.sample_rate)\n    pcm = job / "timeline.pcm"\n    # A sparse file + mmap avoids an in-RAM waveform proportional to video length.\n    with pcm.open("wb") as stream:\n        stream.truncate(total_samples * 2)\n    timeline = np.memmap(pcm, dtype="<i2", mode="r+", shape=(total_samples,))\n    previous_end = 0\n    try:\n        for record in records:\n            samples, rate = sf.read(job / "clips" / record["clip"], dtype="int16")\n            if rate != config.sample_rate or samples.ndim != 1:\n                raise ValueError("Clip does not match mono timeline format")\n            start = record["start_sample"]\n            end = start + len(samples)\n            if start < previous_end or end > total_samples:\n                raise ValueError("Dubbing clip would overlap or exceed the video")\n            timeline[start:end] = samples\n            previous_end = end\n        timeline.flush()\n        output = job / "dub.wav"\n        partial = job / "dub.partial.wav"\n        with wave.open(str(partial), "wb") as stream:\n            stream.setnchannels(1)\n            stream.setsampwidth(2)\n            stream.setframerate(config.sample_rate)\n            for offset in range(0, total_samples, config.sample_rate * 30):\n                stream.writeframesraw(timeline[offset:offset + config.sample_rate * 30].tobytes())\n        os.replace(partial, output)\n        return output\n    finally:\n        del timeline\n        pcm.unlink(missing_ok=True)\n\n\ndef mux_video(local_video, audio, destination, info, config):\n    destination = Path(destination)\n    temp = destination.with_suffix(".partial.mp4")\n    base = ["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",\n            "-i", local_video, "-i", audio, "-map", f"0:{info[\'video_index\']}",\n            "-map", "1:a:0", "-map_metadata", "-1", "-map_chapters", "-1"]\n    suffix = ["-c:a", "aac", "-b:a", "96k", "-metadata:s:a:0", "language=ben",\n              "-t", str(info["duration"]), "-movflags", "+faststart", temp]\n    # H.264 is broadly browser-compatible. Other codecs are explicitly normalized.\n    copied = info["video_codec"] == "h264"\n    if copied:\n        try:\n            run_command(base + ["-c:v", "copy"] + suffix, config.subprocess_timeout)\n        except RuntimeError:\n            LOG.warning("ভিডিও stream-copy হয়নি; H.264-এ encode করা হচ্ছে")\n            copied = False\n    if not copied:\n        run_command(base + ["-c:v", "libx264", "-preset", "veryfast", "-crf", "20",\n                            "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2", "-pix_fmt", "yuv420p",\n                            "-threads", str(config.cpu_threads)] + suffix,\n                    config.subprocess_timeout)\n    output = media_info(temp)\n    audio_streams = [s for s in output["probe"]["streams"] if s["codec_type"] == "audio"]\n    if not audio_streams or abs(output["duration"] - info["duration"]) > 0.25:\n        raise ValueError("Final video duration/audio validation failed")\n    audio_duration = float(audio_streams[0].get("duration", output["duration"]))\n    if abs(audio_duration - info["duration"]) > 0.25:\n        raise ValueError("Final audio does not cover the expected video duration")\n    os.replace(temp, destination)\n    return {"video_stream_copied": copied, "duration": output["duration"]}\n\n\n# %% Checkpointed orchestration and verified publication\n\ndef cached_stage(path, compute: Callable):\n    path = Path(path)\n    if path.exists():\n        LOG.info("আগের checkpoint ব্যবহার হচ্ছে: %s", path.name)\n        return read_json(path)\n    result = compute()\n    atomic_json(path, result)\n    return result\n\n\ndef publish_bundle(job, output_dir, artifacts, manifest):\n    """Manifest-last publication: its presence marks the verified bundle complete."""\n    destination = Path(output_dir) / Path(job).name\n    destination.mkdir(parents=True, exist_ok=True)\n    # Invalidate an earlier completion marker before replacing any bundle files.\n    (destination / "manifest.json").unlink(missing_ok=True)\n    checksums = {}\n    for name in artifacts:\n        checksums[name] = atomic_copy(Path(job) / name, destination / name)\n    completed = {**manifest, "status": "complete", "artifacts_sha256": checksums}\n    atomic_json(Path(job) / "manifest.json", completed)\n    atomic_copy(Path(job) / "manifest.json", destination / "manifest.json")\n    return destination\n\n\ndef process_video(source, config):\n    started = time.perf_counter()\n    job, local_video, source_hash = stage_input(Path(source), config)\n    stage_times = {}\n    current_stage = "probe"\n\n    def timed(name, compute):\n        nonlocal current_stage\n        current_stage = name\n        tick = time.perf_counter()\n        try:\n            return compute()\n        finally:\n            stage_times[name] = round(time.perf_counter() - tick, 3)\n\n    try:\n        info = media_info(local_video)\n        # Estimate PCM/audio disk space before decoding a very long/compressed source.\n        if shutil.disk_usage(job).free < info["duration"] * config.sample_rate * 12 + config.min_free_gb * 1024**3:\n            raise OSError("Insufficient disk for decoded audio/timeline intermediates")\n        if info["duration"] * config.sample_rate * 2 >= 4_000_000_000:\n            raise ValueError("Video exceeds WAV size limit; split it into shorter videos")\n        wav = job / "english.wav"\n        if not (job / "asr.json").exists():\n            timed("extract", lambda: extract_audio(local_video, wav, info, config))\n        asr = timed("asr", lambda: cached_stage(\n            job / "asr.json", lambda: transcribe(wav, info["duration"], config)))\n        validate_segments(asr["segments"], info["duration"])\n        translations = timed("translation", lambda: cached_stage(\n            job / "translation.json", lambda: translate(asr["segments"], config)))\n        validate_segments(translations["segments"], info["duration"], translated=True)\n        if not translations["segments"]:\n            LOG.warning("কথা শনাক্ত হয়নি; ফলাফলে নীরব audio track থাকবে")\n        records, tts_commit = timed("tts", lambda: synthesize(\n            translations["segments"], info["duration"], job, config))\n        audio = timed("timeline", lambda: assemble_timeline(records, info["duration"], job, config))\n        mux = timed("mux", lambda: mux_video(local_video, audio, job / "translated_bn.mp4", info, config))\n        atomic_json(job / "segments.json", records)\n        write_srt(records, "english", job / "english.srt")\n        write_srt(records, "bangla", job / "bangla.srt")\n        manifest = {"pipeline_version": PIPELINE_VERSION, "source_name": Path(source).name,\n                    "source_sha256": source_hash, "config": asdict(config),\n                    "packages": package_versions(), "source_duration": info["duration"],\n                    "language_pair": "eng_Latn -> ben_Beng", "segment_count": len(records),\n                    "no_speech_detected": not bool(records),\n                    "review_required_segments": [r["id"] for r in records if r["review_required"]],\n                    "model_commits": {"asr": asr["model_commit"],\n                                      "translation": translations["model_commit"], "tts": tts_commit},\n                    "stages_seconds_this_run": stage_times,\n                    "processing_seconds_before_publish": round(time.perf_counter() - started, 3),\n                    "mux": mux}\n        destination = timed("publish", lambda: publish_bundle(\n            job, config.output_dir,\n            ["translated_bn.mp4", "english.srt", "bangla.srt", "segments.json", "asr.json"], manifest))\n        result = {"source": str(source), "status": "complete", "output": str(destination),\n                  "review_required_segments": manifest["review_required_segments"],\n                  "seconds": round(time.perf_counter() - started, 3)}\n        if not config.keep_local:\n            shutil.rmtree(job)  # job is always a generated child of the local work directory.\n        return result\n    except Exception as error:\n        atomic_json(job / "failure.json", {"status": "failed", "stage": current_stage,\n                    "error": f"{type(error).__name__}: {error}", "source": str(source),\n                    "stages_seconds_this_run": stage_times})\n        LOG.error("%s ধাপে সমস্যা। Checkpoint রাখা হয়েছে: %s", current_stage, job)\n        raise\n    finally:\n        release_memory()\n\n\ndef run_batch(config):\n    config.validate()\n    for executable in ("ffmpeg", "ffprobe"):\n        if not shutil.which(executable):\n            raise RuntimeError(f"{executable} is missing; run the notebook installation cell")\n    input_dir = Path(config.input_dir)\n    if not input_dir.is_dir():\n        raise FileNotFoundError(f"Create {input_dir} and upload English videos before running")\n    sources = sorted(p for p in input_dir.iterdir()\n                     if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)\n    if not sources:\n        raise FileNotFoundError(f"No supported videos in {input_dir} (scan is non-recursive)")\n    Path(config.output_dir).mkdir(parents=True, exist_ok=True)\n    results = []\n    for source in sources:\n        LOG.info("কাজ চলছে: %s", source.name)\n        try:\n            results.append(process_video(source, config))\n        except Exception as error:\n            LOG.exception("অনুবাদ হয়নি: %s", source.name)\n            results.append({"source": str(source), "status": "failed",\n                            "error": f"{type(error).__name__}: {error}"})\n            if not config.continue_on_error:\n                atomic_json(Path(config.output_dir) / "batch_results.json", results)\n                raise\n    atomic_json(Path(config.output_dir) / "batch_results.json", results)\n    return results\n', 'worker.py': '"""Notebook-এর আলাদা process-এ batch চালায়; kernel-এ কোনো ML model থাকে না।"""\nimport argparse\nimport json\nimport logging\nfrom pathlib import Path\n\nif __package__:\n    from .pipeline import Config, atomic_json, run_batch\nelse:\n    # Standalone notebook একই folder-এ এই দুই source file লেখে।\n    from pipeline import Config, atomic_json, run_batch\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    parser.add_argument("--results", required=True)\n    args = parser.parse_args()\n    logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")\n    try:\n        config = Config(**json.loads(Path(args.config).read_text(encoding="utf-8")))\n        results = run_batch(config)\n        failed = any(item["status"] == "failed" for item in results)\n        outcome = {"status": "failed" if failed else "complete", "results": results}\n    except Exception as error:\n        logging.exception("কাজ সম্পন্ন হয়নি")\n        outcome = {"status": "failed", "results": [],\n                   "error": f"{type(error).__name__}: {error}"}\n    atomic_json(args.results, outcome)\n    return 1 if outcome["status"] == "failed" else 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'}
for name, source in SOURCES.items():
    (APP_DIR / name).write_text(source, encoding="utf-8")
# run_path প্রতি run-এ সর্বশেষ helper পড়ে; kernel-এ torch/transformers import হয় না।
RUNTIME = runpy.run_path(str(APP_DIR / "colab_runtime.py"))
WORKER_PYTHON = RUNTIME["setup_environment"](APP_DIR, '# মূল inference প্যাকেজ নির্দিষ্ট সংস্করণে; Colab-এর torch/CUDA অপরিবর্তিত থাকে।\n# নিজস্ব venv-এ install হয়। Python >= 3.10 এবং torch >= 2.6 প্রয়োজন।\nfaster-whisper==1.2.1\nctranslate2==4.6.0\ntransformers==4.57.6\nsentencepiece==0.2.1\nuroman==1.3.1.1\nsoundfile==0.13.1\nlibrosa==0.11.0\nnumpy>=1.26,<3\nhuggingface-hub>=0.34,<1\nsafetensors>=0.4.3,<1\n')
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
print("Setup প্রস্তুত।")
# শুধু Drive সংযোগ নষ্ট হলে REMOUNT_DRIVE=True করুন।
DRIVE_ROOT = RUNTIME["ensure_drive_mounted"](force_remount=REMOUNT_DRIVE)
INPUT_DIR = DRIVE_ROOT / "Video" / "input"
OUTPUT_DIR = DRIVE_ROOT / "Video" / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive প্রস্তুত। ফলাফল পাবেন: My Drive / Video / output")
CONFIG = {
    "input_dir": str(INPUT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "work_dir": "/content/proxmax-work",
    "device": "auto",                 # প্রয়োজনে "cpu"
    "translation_batch_size": 4,
    "overflow_policy": "stretch",     # বেশি দ্রুত কণ্ঠ হলে ফলাফলে চিহ্ন থাকবে
    "comfortable_speed": 1.50,
    "keep_local": False,               # সফল কাজের checkpoint রাখতে True
    "continue_on_error": True,
}
print("ভিডিও অনুবাদ শুরু হচ্ছে…")
try:
    RUN_RESULT = RUNTIME["run_isolated"](
        WORKER_PYTHON, APP_DIR / "worker.py", CONFIG, APP_DIR / "runs"
    )
except KeyboardInterrupt:
    print("কাজ বন্ধ করা হয়েছে। Worker-ও বন্ধ হয়েছে; আবার এই cell চালাতে পারেন।")

if RUN_RESULT is not None:
    for result in RUN_RESULT["results"]:
        if result["status"] == "complete":
            print("সম্পন্ন:", result["output"])
            if result.get("review_required_segments"):
                print("দ্রুত উচ্চারণের অংশগুলো শুনে দেখুন:", result["review_required_segments"])
        else:
            print("অসম্পূর্ণ:", Path(result["source"]).name, "—", result.get("error", ""))
    if RUN_RESULT["status"] != "complete":
        print(RUN_RESULT.get("error", "কিছু ভিডিও সম্পন্ন হয়নি। উপরের error দেখুন।"))
        print("বিস্তারিত log:", RUN_RESULT["log"])
    else:
        print("সব ভিডিও Drive-এ সংরক্ষিত। প্রকাশের আগে বাংলা অডিও ও subtitle যাচাই করুন।")
